# GNN topology comparison

Complementary fold-safe GNN experiments comparing random, geographic, and training-derived feature-similarity connectivity. The feature-similarity graph is **not** a mutual-information graph.

Set `SEISMIC_DATA_DIR` to the directory containing `features_socal_full.csv` and `full_stations.csv`. Outputs are written to `SEISMIC_GNN_OUTPUT_DIR` when defined, otherwise to `../results/generated/gnn`.


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

Cell 2 — Imports

In [ ]:
# %% Cell 2 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform, pdist
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import kneighbors_graph
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.loader import DataLoader as PyGDataLoader

print(f"PyG: {torch_geometric.__version__}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Cell 3 — Configuration + Load data

In [ ]:
# %% Cell 3 — Configuration + Load data
import os

DATA_DIR = Path(os.environ.get("SEISMIC_DATA_DIR", "../data/processed"))
OUT_DIR = Path(os.environ.get("SEISMIC_GNN_OUTPUT_DIR", "../results/generated/gnn"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
K_GEO = 5
CHANNELS = ['E', 'N', 'Z']
KEEP_FEATURES = [
    'rms', 'peak', 'crest_factor',
    'spectral_bandwidth', 'spectral_centroid', 'spectral_entropy',
    'zcr', 'energy_ratio_early_late',
    'band_5_10Hz', 'band_20_50Hz'
]

df = pd.read_csv(DATA_DIR / "features_socal_full.csv")
stations = pd.read_csv(DATA_DIR / "full_stations.csv")
station_names = stations['receiver_code'].values
N_STATIONS = len(station_names)
station_to_idx = {name: i for i, name in enumerate(station_names)}

# Feature columns
feat_cols = [f'{feat}_{ch}' for ch in CHANNELS for feat in KEEP_FEATURES]
feat_cols = [c for c in feat_cols if c in df.columns]
N_FEATURES = len(feat_cols)

# Filter to selected stations + events ≥2 stations
df = df[df['receiver_code'].isin(set(station_names))].copy()
ev_counts = df.groupby('source_id')['receiver_code'].nunique()
df = df[df['source_id'].isin(ev_counts[ev_counts >= 2].index)].copy()

# Clean features
for col in feat_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

missing = df[feat_cols].isna().sum().sum()
if missing != 0:
    raise ValueError(
        f"Expected complete final feature table, but found {missing} missing values."
    )

n_events = df['source_id'].nunique()
print(f"Stations: {N_STATIONS}, Events: {n_events:,}, Traces: {len(df):,}")
print(f"Features per node: {N_FEATURES}")

Cell 4 — Build geographic + random graphs

In [ ]:
# %% Cell 4 — Build geographic + random graphs
R_EARTH = 6371.0
coords_deg = stations[['lat', 'lon']].values
lat_ref, lon_ref = coords_deg[:, 0].mean(), coords_deg[:, 1].mean()
coords_km = np.column_stack([
    (coords_deg[:, 1] - lon_ref) * np.cos(np.radians(lat_ref)) * np.pi / 180 * R_EARTH,
    (coords_deg[:, 0] - lat_ref) * np.pi / 180 * R_EARTH
])

# Geographic k-NN
A_geo = kneighbors_graph(coords_km, n_neighbors=K_GEO, mode='connectivity', include_self=False)
A_geo = ((A_geo + A_geo.T) > 0).astype(float)
geo_edges = np.array(A_geo.nonzero())
N_GEO_EDGES = geo_edges.shape[1]

# Random graph (same edge count)
def build_random_edges(n_nodes, n_edges, seed=42):
    rng = np.random.RandomState(seed)
    edges = set()
    while len(edges) < n_edges // 2:
        i, j = rng.randint(0, n_nodes, 2)
        if i != j and (i, j) not in edges and (j, i) not in edges:
            edges.add((i, j))
    src, tgt = [], []
    for i, j in edges:
        src.extend([i, j])
        tgt.extend([j, i])
    return np.array([src, tgt])

random_edges = build_random_edges(N_STATIONS, N_GEO_EDGES, seed=SEED)

print(f"Geo graph: {N_GEO_EDGES} directed edges ({N_GEO_EDGES//2} undirected)")
print(f"Random graph: {random_edges.shape[1]} directed edges")

Cell 5 — Feature-similarity graph builder (training fold only)

In [ ]:
# %% Cell 5 — Feature-similarity graph builder (training fold only)

def build_feature_similarity_graph(
    df_train_raw,
    station_to_idx,
    n_stations,
    n_target_edges,
    feat_cols,
    min_observations=20
):
    """
    Build a station feature-similarity graph using TRAINING EVENTS ONLY.

    For each station:
        representation =
        [mean(feature_1), ..., mean(feature_p),
         std(feature_1),  ..., std(feature_p)]

    Station representations are standardized across active stations.
    Pairwise similarity is then defined as

        similarity = 1 - correlation_distance

    and the most similar station pairs are retained until the number
    of directed edges matches n_target_edges.

    IMPORTANT:
    This is NOT a mutual-information graph.
    """

    station_features = {}

    for station, group in df_train_raw.groupby('receiver_code'):

        if station not in station_to_idx:
            continue

        idx = station_to_idx[station]

        feats = group[feat_cols].to_numpy(dtype=np.float64)

        if len(feats) >= min_observations:
            station_features[idx] = feats

    active_stations = sorted(station_features.keys())
    n_active = len(active_stations)

    if n_active < 5:
        return None

    # --------------------------------------------------------
    # Station-level representations: mean + std
    # --------------------------------------------------------

    mean_reps = np.zeros(
        (n_stations, len(feat_cols)),
        dtype=np.float64
    )

    std_reps = np.zeros(
        (n_stations, len(feat_cols)),
        dtype=np.float64
    )

    for idx in active_stations:

        Xs = station_features[idx]

        mean_reps[idx] = Xs.mean(axis=0)
        std_reps[idx] = Xs.std(axis=0)

    reps = np.hstack([
        mean_reps,
        std_reps
    ])

    active_reps = reps[active_stations]

    # Standardization is local to the TRAINING-derived
    # station representations.
    rep_scaler = StandardScaler()
    active_reps_s = rep_scaler.fit_transform(active_reps)

    # --------------------------------------------------------
    # Correlation similarity
    # --------------------------------------------------------

    dist_mat = squareform(
        pdist(
            active_reps_s,
            metric='correlation'
        )
    )

    sim_mat = 1.0 - dist_mat

    # Guard against numerical NaN/Inf from degenerate
    # station representations.
    sim_mat = np.nan_to_num(
        sim_mat,
        nan=-np.inf,
        posinf=-np.inf,
        neginf=-np.inf
    )

    np.fill_diagonal(sim_mat, -np.inf)

    # --------------------------------------------------------
    # Match geographic graph edge count
    # --------------------------------------------------------

    n_undirected_target = n_target_edges // 2

    upper_tri_idx = np.triu_indices(
        n_active,
        k=1
    )

    sims = sim_mat[upper_tri_idx]

    # Number of available pairs can theoretically be smaller
    # than the requested target.
    n_keep = min(
        n_undirected_target,
        len(sims)
    )

    top_idx = np.argsort(-sims)[:n_keep]

    src, tgt = [], []

    for pair_idx in top_idx:

        i_local = upper_tri_idx[0][pair_idx]
        j_local = upper_tri_idx[1][pair_idx]

        i_global = active_stations[i_local]
        j_global = active_stations[j_local]

        # Store both directions.
        src.extend([i_global, j_global])
        tgt.extend([j_global, i_global])

    if len(src) == 0:
        return None

    return np.asarray(
        [src, tgt],
        dtype=np.int64
    )


print("Feature-similarity graph builder ready")

Cell 6 — Normalize features + build PyG dataset

In [ ]:
# %% Cell 6 — Fold-safe preprocessing + PyG event construction

from collections import defaultdict


# ============================================================
# Infrastructure adjacency
# ============================================================

def build_adjacency(edges_np):
    """
    Convert a global edge-index array [2, E] into an adjacency dict.
    """

    adj = defaultdict(set)

    for k in range(edges_np.shape[1]):
        i = int(edges_np[0, k])
        j = int(edges_np[1, k])
        adj[i].add(j)

    return adj


# Geographic and random infrastructure graphs are fixed
# independently of event labels/features.
adj_geo = build_adjacency(geo_edges)
adj_random = build_adjacency(random_edges)


# ============================================================
# Fold-safe feature scaling
# ============================================================

def fit_fold_scaler(df_train_raw, feat_cols):
    """
    Fit StandardScaler using station observations from
    TRAINING EVENTS ONLY.
    """

    scaler = StandardScaler()

    scaler.fit(
        df_train_raw[feat_cols].to_numpy(
            dtype=np.float64
        )
    )

    return scaler


def transform_fold_dataframe(df_raw, feat_cols, scaler):
    """
    Apply a training-fitted scaler without refitting.
    """

    df_scaled = df_raw.copy()

    df_scaled.loc[:, feat_cols] = scaler.transform(
        df_raw[feat_cols].to_numpy(
            dtype=np.float64
        )
    )

    return df_scaled


# ============================================================
# Build PyG events from a fold-specific dataframe
# ============================================================

def build_pyg_dataset(
    df_scaled,
    feat_cols,
    station_to_idx,
    adjacency
):
    """
    Build one PyG Data object per earthquake.

    Node features are already standardized with parameters fitted
    exclusively on the corresponding training fold.

    Multiple traces from the same station/event are averaged,
    matching the station-level event representation used in E4.
    """

    data_list = []
    event_ids = []

    for event_id, group in df_scaled.groupby(
        'source_id',
        sort=False
    ):

        # ----------------------------------------------------
        # Aggregate duplicate station/event rows
        # ----------------------------------------------------

        agg_map = {
            c: 'mean'
            for c in feat_cols
        }

        agg_map['source_magnitude'] = 'first'

        g = (
            group
            .groupby(
                'receiver_code',
                as_index=False
            )
            .agg(agg_map)
        )

        active = {}

        for _, row in g.iterrows():

            station = row['receiver_code']

            if station not in station_to_idx:
                continue

            global_idx = station_to_idx[station]

            active[global_idx] = (
                row[feat_cols]
                .to_numpy(dtype=np.float32)
            )

        if len(active) < 2:
            continue

        # ----------------------------------------------------
        # Global → local node mapping
        # ----------------------------------------------------

        global_nodes = sorted(active.keys())

        local_map = {
            global_idx: local_idx
            for local_idx, global_idx
            in enumerate(global_nodes)
        }

        n_local = len(global_nodes)

        # ----------------------------------------------------
        # Node feature matrix
        # ----------------------------------------------------

        x = torch.zeros(
            (n_local, len(feat_cols)),
            dtype=torch.float32
        )

        for global_idx in global_nodes:

            local_idx = local_map[global_idx]

            x[local_idx] = torch.from_numpy(
                active[global_idx]
            )

        # ----------------------------------------------------
        # Induced event-level edges
        # ----------------------------------------------------

        src = []
        tgt = []

        for global_i in global_nodes:

            for global_j in adjacency.get(
                global_i,
                set()
            ):

                if global_j in local_map:

                    src.append(
                        local_map[global_i]
                    )

                    tgt.append(
                        local_map[global_j]
                    )

        # Preserve original notebook behavior:
        # if the induced infrastructure graph has no edge,
        # use a complete directed event graph.
        if len(src) == 0:

            for i in range(n_local):
                for j in range(n_local):

                    if i != j:
                        src.append(i)
                        tgt.append(j)

        edge_index = torch.tensor(
            [src, tgt],
            dtype=torch.long
        )

        # ----------------------------------------------------
        # Target and mask
        # ----------------------------------------------------

        y = torch.tensor(
            [float(g['source_magnitude'].iloc[0])],
            dtype=torch.float32
        )

        mask = torch.ones(
            n_local,
            dtype=torch.bool
        )

        data = Data(
            x=x,
            edge_index=edge_index,
            y=y,
            mask=mask
        )

        data.event_id = str(event_id)

        data_list.append(data)
        event_ids.append(event_id)

    return data_list, np.asarray(event_ids)


# ============================================================
# Diagnostics
# ============================================================

def graph_diagnostics(data_list):

    n_nodes = np.asarray([
        d.x.size(0)
        for d in data_list
    ])

    n_edges = np.asarray([
        d.edge_index.size(1)
        for d in data_list
    ])

    return {
        'N_events': len(data_list),
        'nodes_mean': n_nodes.mean(),
        'nodes_median': np.median(n_nodes),
        'nodes_max': n_nodes.max(),
        'edges_mean': n_edges.mean(),
        'edges_median': np.median(n_edges),
    }


print("Fold-safe preprocessing and PyG builders ready")
print("No global feature scaling has been applied.")

Cell 7 — Model definitions

In [ ]:
# %% Cell 7 — Model definitions
class GATv2Model(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.convs.append(GATv2Conv(in_dim, hidden_dim, heads=heads,
                                     dropout=dropout, concat=True))
        self.norms.append(nn.BatchNorm1d(hidden_dim * heads))
        for _ in range(n_layers - 1):
            self.convs.append(GATv2Conv(hidden_dim * heads, hidden_dim, heads=heads,
                                         dropout=dropout, concat=True))
            self.norms.append(nn.BatchNorm1d(hidden_dim * heads))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * heads, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, mask, batch = data.x, data.edge_index, data.mask, data.batch
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        # Masked global mean pool
        x_masked = x.clone()
        x_masked[~mask] = 0.0
        out = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        count = torch.zeros(batch.max().item() + 1, 1, device=x.device)
        active_idx = mask.nonzero(as_tuple=True)[0]
        out.scatter_add_(0, batch[active_idx].unsqueeze(1).expand(-1, x.size(1)),
                        x_masked[active_idx])
        count.scatter_add_(0, batch[active_idx].unsqueeze(1),
                          torch.ones(len(active_idx), 1, device=x.device))
        count = count.clamp(min=1)
        out = out / count
        return self.mlp(out).squeeze(-1)


class GCNModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(n_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.norms.append(nn.BatchNorm1d(hidden_dim))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, mask, batch = data.x, data.edge_index, data.mask, data.batch
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x_masked = x.clone()
        x_masked[~mask] = 0.0
        out = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        count = torch.zeros(batch.max().item() + 1, 1, device=x.device)
        active_idx = mask.nonzero(as_tuple=True)[0]
        out.scatter_add_(0, batch[active_idx].unsqueeze(1).expand(-1, x.size(1)),
                        x_masked[active_idx])
        count.scatter_add_(0, batch[active_idx].unsqueeze(1),
                          torch.ones(len(active_idx), 1, device=x.device))
        count = count.clamp(min=1)
        out = out / count
        return self.mlp(out).squeeze(-1)

print("Models defined: GATv2Model, GCNModel")

Cell 8 — Training function

In [ ]:
# %% Cell 8 — Training function
def train_gnn(model, train_data, val_data,
              lr=1e-3, epochs=200, batch_size=128, patience=25):

    train_loader = PyGDataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = PyGDataLoader(val_data, batch_size=batch_size, shuffle=False)

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()

    best_val_loss, best_state, counter = float('inf'), None, 0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            pred = model(batch)
            loss = criterion(pred, batch.y.squeeze())
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_loss, val_n = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                pred = model(batch)
                val_loss += criterion(pred, batch.y.squeeze()).item() * batch.num_graphs
                val_n += batch.num_graphs
        val_loss /= val_n
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    model.load_state_dict(best_state); model.eval(); model = model.to(DEVICE)
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            all_pred.extend(model(batch).cpu().numpy())
            all_true.extend(batch.y.squeeze().cpu().numpy())

    return np.array(all_pred), np.array(all_true), epoch + 1

print("Training function ready")

Cell 9 — Run experiments

In [ ]:
# %% Cell 9 — Run fold-safe GNN experiments

import random
import gc


print("=" * 72)
print("E4 — GNN TOPOLOGY COMPARISON — FOLD-SAFE PREPROCESSING")
print("=" * 72)


# ============================================================
# Event-level folds
# ============================================================

all_event_ids = df['source_id'].drop_duplicates().to_numpy()

gkf = GroupKFold(
    n_splits=N_FOLDS
)

split_indices = list(
    gkf.split(
        X=np.zeros(len(all_event_ids)),
        y=np.zeros(len(all_event_ids)),
        groups=all_event_ids
    )
)


# ============================================================
# Experiments
# ============================================================

EXPERIMENTS = {
    'Random-GATv2':     ('gatv2', 'random'),
    'Geo-GATv2':        ('gatv2', 'geo'),
    'FeatureSim-GATv2': ('gatv2', 'feature_sim'),
    'Geo-GCN':          ('gcn',   'geo'),
}


# Store out-of-fold results
prediction_records = {
    name: []
    for name in EXPERIMENTS
}

fold_metric_records = []


# ============================================================
# Cross-validation
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    split_indices
):

    print("\n" + "=" * 72)
    print(f"FOLD {fold}")
    print("=" * 72)

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    fold_seed = SEED + fold

    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    # --------------------------------------------------------
    # Event split
    # --------------------------------------------------------

    train_events = set(
        all_event_ids[train_idx]
    )

    val_events = set(
        all_event_ids[val_idx]
    )

    df_train_raw = df[
        df['source_id'].isin(train_events)
    ].copy()

    df_val_raw = df[
        df['source_id'].isin(val_events)
    ].copy()

    print(
        f"Train events: "
        f"{df_train_raw['source_id'].nunique():,}"
    )

    print(
        f"Validation events: "
        f"{df_val_raw['source_id'].nunique():,}"
    )

    # --------------------------------------------------------
    # Fit scaler on TRAINING fold only
    # --------------------------------------------------------

    fold_scaler = fit_fold_scaler(
        df_train_raw,
        feat_cols
    )

    df_train = transform_fold_dataframe(
        df_train_raw,
        feat_cols,
        fold_scaler
    )

    df_val = transform_fold_dataframe(
        df_val_raw,
        feat_cols,
        fold_scaler
    )

    # --------------------------------------------------------
    # Sanity check
    # --------------------------------------------------------

    train_scaled = (
        df_train[feat_cols]
        .to_numpy(dtype=np.float64)
    )

    val_scaled = (
        df_val[feat_cols]
        .to_numpy(dtype=np.float64)
    )

    print(
        "Scaled training mean "
        f"(mean |mu_j|): "
        f"{np.abs(train_scaled.mean(axis=0)).mean():.3e}"
    )

    print(
        "Scaled training std "
        f"(mean sigma_j): "
        f"{train_scaled.std(axis=0).mean():.4f}"
    )

    print(
        "Validation scaling uses training statistics only."
    )

    # --------------------------------------------------------
    # Feature-similarity topology:
    # RAW TRAINING DATA ONLY
    # --------------------------------------------------------

    feature_sim_edges = (
        build_feature_similarity_graph(
            df_train_raw=df_train_raw,
            station_to_idx=station_to_idx,
            n_stations=N_STATIONS,
            n_target_edges=N_GEO_EDGES,
            feat_cols=feat_cols,
            min_observations=20
        )
    )

    if feature_sim_edges is None:

        raise RuntimeError(
            f"Fold {fold}: feature-similarity graph "
            "could not be constructed."
        )

    adj_feature_sim = build_adjacency(
        feature_sim_edges
    )

    print(
        "FeatureSim graph: "
        f"{feature_sim_edges.shape[1]} directed edges"
    )

    # --------------------------------------------------------
    # Run all four GNN configurations
    # --------------------------------------------------------

    for exp_idx, (
        exp_name,
        (arch, graph_type)
    ) in enumerate(EXPERIMENTS.items()):

        print("\n" + "-" * 60)
        print(f"{exp_name} — fold {fold}")
        print("-" * 60)

        # Use deterministic but distinct seed for
        # each model/fold combination.
        model_seed = (
            SEED
            + 1000 * fold
            + exp_idx
        )

        random.seed(model_seed)
        np.random.seed(model_seed)
        torch.manual_seed(model_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(
                model_seed
            )

        # ----------------------------------------------------
        # Select infrastructure topology
        # ----------------------------------------------------

        if graph_type == 'geo':

            adjacency = adj_geo

        elif graph_type == 'random':

            adjacency = adj_random

        elif graph_type == 'feature_sim':

            adjacency = adj_feature_sim

        else:

            raise ValueError(
                f"Unknown graph type: {graph_type}"
            )

        # ----------------------------------------------------
        # Build fold-specific PyG objects
        # ----------------------------------------------------

        train_data, train_eids = (
            build_pyg_dataset(
                df_train,
                feat_cols,
                station_to_idx,
                adjacency
            )
        )

        val_data, val_eids = (
            build_pyg_dataset(
                df_val,
                feat_cols,
                station_to_idx,
                adjacency
            )
        )

        print(
            f"PyG events: "
            f"train={len(train_data):,}, "
            f"val={len(val_data):,}"
        )

        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------

        if fold == 0:

            diag = graph_diagnostics(
                val_data
            )

            print(
                "Validation graph diagnostics: "
                f"nodes median="
                f"{diag['nodes_median']:.0f}, "
                f"edges median="
                f"{diag['edges_median']:.0f}"
            )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        if arch == 'gatv2':

            model = GATv2Model(
                N_FEATURES,
                hidden_dim=64,
                heads=4,
                n_layers=2,
                dropout=0.2
            )

        elif arch == 'gcn':

            model = GCNModel(
                N_FEATURES,
                hidden_dim=64,
                n_layers=2,
                dropout=0.2
            )

        else:

            raise ValueError(
                f"Unknown architecture: {arch}"
            )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        pred, true, n_epochs = train_gnn(
            model,
            train_data,
            val_data,
            lr=1e-3,
            epochs=200,
            batch_size=128,
            patience=25
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        mae = mean_absolute_error(
            true,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                true,
                pred
            )
        )

        r2 = r2_score(
            true,
            pred
        )

        print(
            f"Fold {fold}: "
            f"MAE={mae:.4f}, "
            f"RMSE={rmse:.4f}, "
            f"R²={r2:.4f}, "
            f"epochs={n_epochs}"
        )

        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        fold_metric_records.append({
            'Model': exp_name,
            'Fold': fold,
            'N': len(true),
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'Epochs': n_epochs
        })

        # ----------------------------------------------------
        # Store event-level predictions
        # ----------------------------------------------------

        if len(val_eids) != len(pred):

            raise RuntimeError(
                "Prediction/event-ID length mismatch: "
                f"{len(pred)} predictions vs "
                f"{len(val_eids)} event IDs."
            )

        pred_fold_df = pd.DataFrame({
            'event_id': val_eids,
            'y_true': true,
            'y_pred': pred,
            'fold': fold,
            'model': exp_name
        })

        prediction_records[
            exp_name
        ].append(pred_fold_df)

        # ----------------------------------------------------
        # Cleanup GPU memory
        # ----------------------------------------------------

        del model
        del train_data
        del val_data

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# Combine out-of-fold predictions
# ============================================================

results = {}

for exp_name in EXPERIMENTS:

    pred_df = pd.concat(
        prediction_records[exp_name],
        ignore_index=True
    )

    mae = mean_absolute_error(
        pred_df['y_true'],
        pred_df['y_pred']
    )

    rmse = np.sqrt(
        mean_squared_error(
            pred_df['y_true'],
            pred_df['y_pred']
        )
    )

    r2 = r2_score(
        pred_df['y_true'],
        pred_df['y_pred']
    )

    results[exp_name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'predictions': pred_df
    }


# ============================================================
# Save
# ============================================================

fold_metrics = pd.DataFrame(
    fold_metric_records
)

fold_metrics.to_csv(
    OUT_DIR /
    "gnn_fold_metrics_foldsafe.csv",
    index=False
)

all_predictions = pd.concat(
    [
        results[name]['predictions']
        for name in EXPERIMENTS
    ],
    ignore_index=True
)

all_predictions.to_csv(
    OUT_DIR /
    "gnn_predictions_foldsafe.csv",
    index=False
)


# ============================================================
# Overall summary
# ============================================================

print("\n" + "=" * 72)
print("FOLD-SAFE GNN RESULTS")
print("=" * 72)

for exp_name, res in results.items():

    print(
        f"{exp_name:<20s} "
        f"MAE={res['MAE']:.4f}  "
        f"RMSE={res['RMSE']:.4f}  "
        f"R²={res['R²']:.4f}"
    )

print("\nSaved:")
print(
    OUT_DIR /
    "gnn_fold_metrics_foldsafe.csv"
)
print(
    OUT_DIR /
    "gnn_predictions_foldsafe.csv"
)

Cell 10 — Results comparison

In [ ]:
# %% Cell 10 — Final corrected GNN comparison

print("=" * 72)
print("E4 — CORRECTED GNN RESULTS")
print("=" * 72)


# ============================================================
# Corrected GNN table
# ============================================================

comparison_rows = []

for name, res in results.items():

    comparison_rows.append({
        'Model': name,
        'MAE': res['MAE'],
        'RMSE': res['RMSE'],
        'R2': res['R²']
    })


df_comp = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        'MAE',
        ascending=True
    )
    .reset_index(drop=True)
)


print("\nCorrected fold-safe GNN results:\n")

print(
    df_comp.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# Previous values — audit only
# ============================================================

previous_results = pd.DataFrame([
    {
        'Model': 'Random-GATv2',
        'Previous_MAE': 0.1928,
        'Previous_RMSE': 0.2581,
        'Previous_R2': 0.8659
    },
    {
        'Model': 'Geo-GATv2',
        'Previous_MAE': 0.1979,
        'Previous_RMSE': 0.2633,
        'Previous_R2': 0.8605
    },
    {
        'Model': 'FeatureSim-GATv2',
        'Previous_MAE': 0.1974,
        'Previous_RMSE': 0.2611,
        'Previous_R2': 0.8628
    },
    {
        'Model': 'Geo-GCN',
        'Previous_MAE': 0.2228,
        'Previous_RMSE': 0.2947,
        'Previous_R2': 0.8252
    }
])


audit = df_comp.merge(
    previous_results,
    on='Model',
    how='left'
)

audit['Delta_MAE_corrected_minus_previous'] = (
    audit['MAE']
    - audit['Previous_MAE']
)

audit['Delta_RMSE_corrected_minus_previous'] = (
    audit['RMSE']
    - audit['Previous_RMSE']
)

audit['Delta_R2_corrected_minus_previous'] = (
    audit['R2']
    - audit['Previous_R2']
)


print("\nChange relative to original globally scaled run:\n")

print(
    audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


# ============================================================
# Fold table
# ============================================================

print("\nFold-level corrected results:\n")

print(
    fold_metrics
    .sort_values(
        ['Model', 'Fold']
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# Save final tables
# ============================================================

df_comp.to_csv(
    OUT_DIR /
    "gnn_overall_metrics_foldsafe.csv",
    index=False
)

audit.to_csv(
    OUT_DIR /
    "gnn_scaling_correction_audit.csv",
    index=False
)

print("\nSaved final tables to:")
print(OUT_DIR)